## Logistic regression

Logistic regression is classification method which is a bit similar to linear regression, but used for categorical dependent variable. Instead of predicting a value of the target variable, which is assumed to be linear, logistic regression tries to predict the probabilities of a binary response. Some common examples are:

* marking an email as a spam
* marking given medical image as possible illness

For the classification we use a logistic function:

\begin{align}
p = \frac{1}{1 + e^{-(b_{0} + b_{1}x)}}
\end{align}

Let's consider the same dataset like in the example of linear regression. This time, we are not going to predict an exact price, but instead would like to know if given zone is expensive when it comes to a median of value. First of all, we need to consider which examples should be thought to be expensive.

In [1]:
# Show some statistics about the dataset
from sklearn.datasets import load_boston
import pandas as pd
import numpy as np

# Loading the dataset with pandas
boston_data = load_boston()
boston_housing_df = pd.DataFrame(boston_data.data,columns=boston_data.feature_names)
boston_housing_df["MEDV"] = boston_data.target
boston_housing_df.head()

boston_housing_df.describe().T

ImportError: 
`load_boston` has been removed from scikit-learn since version 1.2.

The Boston housing prices dataset has an ethical problem: as
investigated in [1], the authors of this dataset engineered a
non-invertible variable "B" assuming that racial self-segregation had a
positive impact on house prices [2]. Furthermore the goal of the
research that led to the creation of this dataset was to study the
impact of air quality but it did not give adequate demonstration of the
validity of this assumption.

The scikit-learn maintainers therefore strongly discourage the use of
this dataset unless the purpose of the code is to study and educate
about ethical issues in data science and machine learning.

In this special case, you can fetch the dataset from the original
source::

    import pandas as pd
    import numpy as np

    data_url = "http://lib.stat.cmu.edu/datasets/boston"
    raw_df = pd.read_csv(data_url, sep="\s+", skiprows=22, header=None)
    data = np.hstack([raw_df.values[:

A mean of *medv* is 22.768769. By a simple thresholding we'll add another feature to our dataset, called **is_expensive** which will be set to 1 whenever the median value is higher than the mean.

In [ ]:
# Get the mean value of the medv as a threshold
IS_EXPENSIVE_THRESHOLD = boston_housing_df["MEDV"].mean()

# Append a new column to the dataset
boston_housing_df["is_expensive"] = boston_housing_df["MEDV"]\
    .map(lambda x: int(x > IS_EXPENSIVE_THRESHOLD))
boston_housing_df.head()

The perfect predictor for the *is_expensive* variable is naturally *medv* feature, but we are not going to consider it. In one of the previous examples we found out that column *lstat* has the highest absolute correlation with *medv*, so for the 2D example we will consider it as an independent variable. 

### 2D logistic regression

For the intuition, we are going to plot the *lstat* feature with *is_expensive* first, and then apply logistic regression to find, what is a threshold found by the algorithm itself.

In [ ]:
# Display a scatter plot: lstat vs is_expensive
boston_housing_df.plot.scatter(x="LSTAT", y="is_expensive", color="blue")

In [ ]:
from sklearn.linear_model import LogisticRegression

# Create an instance of LogisticRegression and fit it
logistic_regression = LogisticRegression()
logistic_regression.fit(X=boston_housing_df[["LSTAT"]], 
                        y=boston_housing_df["is_expensive"])

# Check the coefficients of the created model
logistic_regression.coef_, logistic_regression.intercept_

In [ ]:
import matplotlib.pyplot as plt

# Calculating the value of the logistic function
x_values = np.linspace(0, 40)
y_values = 1.0 / (1.0 + np.exp(-(logistic_regression.coef_[0] * x_values + logistic_regression.intercept_[0])))

# Display a scatter plot: lstat vs is_expensive and logistic function
boston_housing_df.plot(x="LSTAT", y="is_expensive", color="blue", kind="scatter")
plt.plot(x_values, y_values, color="red", linestyle="dashed")

With a simple thresholing we may perform a classification - if the *is_expensive* value returned by the classifier is higher than 50%, we can consider it to be expensive.

### Multidimensional logistic regression

Once again, we will consider an example with two independent variables, in order to visualize the predictions in the 3-dimensional space. As the most correlated values are typically the best predictors for the linear models, we are going to consider *lstat* together with *rm*.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Scale the original dataset first, and then run regression
scaler = StandardScaler()
boston_housing_lstatrm_scaled = scaler.fit_transform(
    boston_housing_df[["LSTAT", "RM"]])

# Create an instance of LogisticRegression and fit it
logistic_regression = LogisticRegression()
logistic_regression.fit(X=boston_housing_lstatrm_scaled, 
                        y=boston_housing_df["is_expensive"])

# Check the coefficients of the created model
logistic_regression.coef_, logistic_regression.intercept_

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

# Save the coefficients in a single array
coefficients = np.append(logistic_regression.coef_[0],
                         logistic_regression.intercept_[0])

# Calculate the values for a selected range
x = np.linspace(-2, 2)
y = np.linspace(-2, 2)
x_values, y_values = np.meshgrid(x, y)
z_values = 1.0 / (1.0 + np.exp(-(coefficients[0] * x_values + coefficients[1] * y_values + coefficients[2])))

# Display 3D scatter: rm, lstat vs is_expensive and logistic function
fig = plt.figure()
ax = fig.add_subplot(111, projection="3d")
ax.scatter(boston_housing_lstatrm_scaled[:, 0], 
           boston_housing_lstatrm_scaled[:, 1], 
           boston_housing_df["is_expensive"], c="blue")
ax.plot_surface(x_values, y_values, z_values, linewidth=0.2, 
                color="red", alpha=0.5)
plt.show()